# Modules, packages and working with files
* until now every example lived inside one single notebook/script
* real projects grow too big for one file, python solves this with modules and packages
* we also learn how to read/write files - the very first step to working with real data

## modules
* a module is nothing more than a single .py file containing python code (functions, classes, variables)
* we already used modules without noticing: every "import xyz" statement loads a module
* python ships with a huge standard library of modules that are always available, no pip install needed

In [1]:
# importing a whole module, we then need to prefix everything with the modulename and a dot
import math

print(math.pi)
print(math.sqrt(64))
print(math.floor(4.7), math.ceil(4.2))

# importing only specific names out of a module - no prefix needed anymore, but naming clashes with your own code become more likely
from math import pi, sqrt

print(pi, sqrt(81))

# importing a module with an alias - very common for long or often-used module names
import math as m
print(m.pi)

# importing everything from a module with "*" is possible but strongly discouraged in real projects
# it pollutes your namespace and it becomes unclear where a function actually came from
# from math import *  # avoid this in production code!

3.141592653589793
8.0
4 5
3.141592653589793 9.0
3.141592653589793


In [2]:
# some of the most used standard library modules, just a short preview - explore the rest yourself with help(modulename)
import random
import datetime
import os
import sys as system_module

print("random int 1-10:", random.randint(1, 10))
print("random choice:", random.choice(["rock", "paper", "scissors"]))

print("today:", datetime.date.today())
print("now:", datetime.datetime.now())

print("current working directory:", os.getcwd())
print("python version:", system_module.version.split()[0])

# u are able to ask any module what it contains with the builtin function dir()
print("a few things inside random module:", dir(random)[:8])

random int 1-10: 1
random choice: paper
today: 2026-09-26
now: 2026-09-26 10:52:29.126019
current working directory: /home/user/education_absosol_python
python version: 3.11.15
a few things inside random module: ['BPF', 'LOG4', 'NV_MAGICCONST', 'RECIP_BPF', 'Random', 'SG_MAGICCONST', 'SystemRandom', 'TWOPI']


## packages
* a package is a folder containing multiple module files, used to group related modules together
* historically a folder needed a special (often empty) file called `__init__.py` to be recognized as a package
* since python 3.3, "namespace packages" also work without `__init__.py`, but adding one is still very common and recommended for clarity
* example folder structure of a package called "shapes":

In [3]:
example_structure = """
myproject/
    main.py
    shapes/
        __init__.py
        circle.py
        square.py
"""
print(example_structure)

# inside main.py you would then write:
#     from shapes import circle
#     from shapes.circle import area
# or, if shapes/__init__.py contains "from .circle import area":
#     from shapes import area

# expert note: the dot inside "from .circle import area" (used INSIDE a package) means "relative import from the current package"
# this only works inside a real package, not inside a single standalone script


myproject/
    main.py
    shapes/
        __init__.py
        circle.py
        square.py



## working with files
* to read or write real files (text, logs, config, csv, ...) python provides the builtin `open()` function
* the recommended way to open a file is the "with" statement (a context manager) - it automatically closes the file afterwards, even if an error happens inside the block

In [4]:
# writing text to a file
# mode "w" = write (overwrites the whole file if it already exists!)
# mode "a" = append (adds to the end of an existing file, keeps the rest)
# mode "r" = read (default mode, file must already exist)

with open("demo_file.txt", "w") as file:
    file.write("first line\n")
    file.write("second line\n")

with open("demo_file.txt", "a") as file:
    file.write("third line, appended later\n")

# after the "with" block ends, the file is automatically closed - we dont need to call file.close() ourselves
print("file written")

file written


In [5]:
# reading a file back, again with "with" so it closes automatically
with open("demo_file.txt", "r") as file:
    content = file.read()  # reads the WHOLE file as one single string
print("full content:\n", content)

# reading line by line is often more useful, a file object is directly iterable
with open("demo_file.txt", "r") as file:
    for line in file:
        print("line:", line.strip())  # .strip() removes the trailing newline character

# .readlines() reads all lines at once into a list of strings
with open("demo_file.txt", "r") as file:
    all_lines = file.readlines()
print("as a list:", all_lines)

full content:
 first line
second line
third line, appended later

line: first line
line: second line
line: third line, appended later
as a list: ['first line\n', 'second line\n', 'third line, appended later\n']


In [6]:
# small cleanup - remove the demo file we created, using the os module
import os
if os.path.exists("demo_file.txt"):
    os.remove("demo_file.txt")
    print("demo_file.txt removed again")

demo_file.txt removed again


## pathlib - the modern way to work with file paths
* the classic way to build file paths used plain strings and `os.path.join()`
* since python 3.4 the `pathlib` module offers an object-oriented, more readable way to work with paths, and it automatically handles the differences between windows ("\\") and linux/mac ("/") path separators

In [7]:
from pathlib import Path

current_dir = Path(".")
print("current directory:", current_dir.resolve())

demo_path = Path("some_folder") / "some_subfolder" / "file.txt"  # the "/" operator builds a path, no manual string concatenation needed
print("built path:", demo_path)
print("just the filename:", demo_path.name)
print("just the extension:", demo_path.suffix)
print("just the parent folder:", demo_path.parent)

# pathlib also has convenience methods for reading/writing small text files without an explicit "with" block
readme_path = Path("README.md")
if readme_path.exists():
    print("README.md exists, first 60 characters:", readme_path.read_text()[:60])

current directory: /home/user/education_absosol_python
built path: some_folder/some_subfolder/file.txt
just the filename: file.txt
just the extension: .txt
just the parent folder: some_folder/some_subfolder
README.md exists, first 60 characters: # donation
* if you enjoy my stuff and want to make me happy


## working with structured data: json and csv
* real applications rarely work with plain unstructured text, JSON and CSV are two of the most common structured data formats
* both have a dedicated, always-available standard library module

In [8]:
import json

# a normal python dictionary
person = {"name": "Alex", "age": 30, "hobbies": ["chess", "python"], "active": True}

# json.dumps() converts a python object into a JSON formatted string
json_text = json.dumps(person, indent=2)
print(json_text)
print(type(json_text))

# json.loads() converts a JSON formatted string back into a python object
parsed_back = json.loads(json_text)
print(parsed_back, type(parsed_back))
print(parsed_back["hobbies"])

# json.dump()/json.load() (without the trailing "s") work directly with an open file object instead of a string
with open("demo_person.json", "w") as file:
    json.dump(person, file, indent=2)

with open("demo_person.json", "r") as file:
    loaded_person = json.load(file)
print("loaded from file:", loaded_person)

import os
os.remove("demo_person.json")

{
  "name": "Alex",
  "age": 30,
  "hobbies": [
    "chess",
    "python"
  ],
  "active": true
}
<class 'str'>
{'name': 'Alex', 'age': 30, 'hobbies': ['chess', 'python'], 'active': True} <class 'dict'>
['chess', 'python']
loaded from file: {'name': 'Alex', 'age': 30, 'hobbies': ['chess', 'python'], 'active': True}


In [9]:
import csv

# writing rows to a csv file
rows = [
    ["name", "age", "city"],
    ["Alex", 30, "Berlin"],
    ["Mia", 22, "Hamburg"],
]

with open("demo_people.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

# reading rows back
with open("demo_people.csv", "r", newline="") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

# csv.DictReader reads each row directly as a dictionary, using the first row as keys - very convenient for real data
with open("demo_people.csv", "r", newline="") as file:
    dict_reader = csv.DictReader(file)
    for row in dict_reader:
        print(row["name"], "is", row["age"], "years old and lives in", row["city"])

import os
os.remove("demo_people.csv")

['name', 'age', 'city']
['Alex', '30', 'Berlin']
['Mia', '22', 'Hamburg']
Alex is 30 years old and lives in Berlin
Mia is 22 years old and lives in Hamburg


## quick comparison to other languages
* python's `import module` / `from module import name` is comparable to JavaScript's `import ... from "module"` (ES modules) or `require("module")` (Node.js CommonJS), Java's `import package.Class;`, Rust's `use module::name;`, Go's `import "package"`
* a python "package" (a folder of modules) is the same idea as a JavaScript npm package, a Java package (folder matching the package declaration), or a Rust crate/module
* python's standard library is famous for being "batteries included" - modules like json, csv, datetime, os, math ship with every python installation; JavaScript/Node.js by comparison ships a much smaller core and relies heavily on npm packages for things python already has built in
* the `with open(...) as file:` context manager pattern is python's equivalent of Java/C#'s `try-with-resources` / `using` statement, or (loosely) RAII destructors in C++/Rust - all of them guarantee cleanup (closing a file/connection) even if an error occurs
* JSON is a first-class citizen in JavaScript (`JSON.parse()`/`JSON.stringify()` are nearly identical in naming and behaviour to python's `json.loads()`/`json.dumps()`), while statically typed languages like Java/C#/Rust usually need an external library (Jackson/Gson, System.Text.Json, serde) to map JSON into typed objects

## Exercises
1. write a small script that creates a file `notes.txt`, writes 3 lines of your choice into it, then reads the file back and prints each line with its line number (use `enumerate()`)
2. create a python dictionary describing a small inventory (item names as keys, item counts as values), save it as `inventory.json`, then load it back and print the total amount of items (sum of all values)
3. create a csv file `scores.csv` with columns `player` and `score` for at least 3 players, then read it back with `csv.DictReader` and print the name of the player with the highest score
4. use `pathlib.Path` to build a path `data/2024/report.csv` without ever using a plain "/" string concatenation, and print its `.parent`, `.name` and `.suffix`

In [10]:
# TODO: solve exercise 1 here (notes.txt write + read with line numbers)


# TODO: solve exercise 2 here (inventory.json)


# TODO: solve exercise 3 here (scores.csv, highest score)


# TODO: solve exercise 4 here (pathlib path building)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
import os, json, csv
from pathlib import Path

# sample solution 1
with open("notes.txt", "w") as file:
    file.write("first note\n")
    file.write("second note\n")
    file.write("third note\n")

with open("notes.txt", "r") as file:
    for line_number, line in enumerate(file, start=1):
        print(line_number, line.strip())
os.remove("notes.txt")

# sample solution 2
inventory = {"apples": 10, "bananas": 5, "oranges": 8}
with open("inventory.json", "w") as file:
    json.dump(inventory, file)
with open("inventory.json", "r") as file:
    loaded_inventory = json.load(file)
print("total items:", sum(loaded_inventory.values()))
os.remove("inventory.json")

# sample solution 3
with open("scores.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["player", "score"])
    writer.writerow(["Alex", 42])
    writer.writerow(["Mia", 99])
    writer.writerow(["Sam", 17])

with open("scores.csv", "r", newline="") as file:
    reader = list(csv.DictReader(file))
best_player = max(reader, key=lambda row: int(row["score"]))
print("highest score:", best_player["player"], best_player["score"])
os.remove("scores.csv")

# sample solution 4
report_path = Path("data") / "2024" / "report.csv"
print(report_path, "-->", report_path.parent, report_path.name, report_path.suffix)

1 first note
2 second note
3 third note
total items: 23
highest score: Mia 99
data/2024/report.csv --> data/2024 report.csv .csv


* congratulation you finished this lesson

## what we learned
* importing standard library modules in different styles (import, from...import, aliasing)
* the basic structure of a package with `__init__.py`
* reading and writing files with `open()` and the `with` context manager
* the modern `pathlib` module for building file paths
* working with structured data formats: `json` and `csv`